In [1]:
### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime
import h5py
import numpy as np
np.set_printoptions(suppress=True)


## test for single file

In [1]:
from pyhdf.SD import SD, SDC

fn = '/data/shared_data/satellite_data/CloudSat/2B-GEOPROF-LIDAR.P2_R05/2006/163/2006163205526_00663_CS_2B-GEOPROF-LIDAR_GRANULE_P2_R05_E00_F00.hdf'
hdf = SD(fn, SDC.READ)

In [1]:
from pyhdf.SD import SD, SDC

fn = '/data/shared_data/satellite_data/CloudSat/2B-CWC-RVOD.P1_R05/2006/163/2006163205526_00663_CS_2B-CWC-RVOD_GRANULE_P1_R05_E00_F00.hdf'
hdf = SD(fn, SDC.READ)

datasets = hdf.datasets()

print("Total variables:", len(datasets))
print("\nAll variable names:\n")

for name in datasets.keys():
    print(name)

Total variables: 13

All variable names:

Height
Liq_Water_Content
Liq_Water_Content_Uncert
Cloud_Liq_Water_Content
Precip_Liq_Water_Content
Liq_Geom_Mean_Radius
Liq_Geom_Mean_Radius_Uncert
Liq_Number_Concentration
Liq_Number_Concentration_Uncert
Ice_Water_Content
Ice_Water_Content_Uncert
Phase
Radar_Reflectivity_Fwd


## Loop for each product

In [8]:
import os, glob
import numpy as np
import xarray as xr
from pyhdf.SD import SD, SDC
from pyhdf.HDF import HDF
from datetime import datetime, timedelta

SRC_ROOT = "/data/shared_data/satellite_data/CloudSat/2B-GEOPROF-LIDAR.P2_R05"
OUT_ENA  = "/data/ggong/CloudSat/2B-GEOPROF-LIDAR.P2_R05/ENA_5x5"
OUT_SGP  = "/data/ggong/CloudSat/2B-GEOPROF-LIDAR.P2_R05/SGP_5x5"

ENA = (36.5916, 41.5916, -30.5257, -25.5257)
SGP = (34.107322, 39.107322, -99.987643, -94.987643)

os.makedirs(OUT_ENA, exist_ok=True)
os.makedirs(OUT_SGP, exist_ok=True)

def read_vdata_by_ref(fn, ref, field):
    h = HDF(fn); vs = h.vstart()
    v = vs.attach(ref)
    nrecs, _, _, _, _ = v.inquire()
    v.setfields(field); raw = v.read(nrecs)
    v.detach(); vs.end(); h.close()
    arr = np.array(raw)
    return arr[:, 0] if (arr.ndim == 2 and arr.shape[1] == 1) else arr

def t0_from_filename(fn):
    # filename begins with YYYYDOYHHMMSS, e.g. 2006163205526
    stem = os.path.basename(fn)[:13]
    y, doy = int(stem[0:4]), int(stem[4:7])
    hh, mm, ss = int(stem[7:9]), int(stem[9:11]), int(stem[11:13])
    return datetime(y, 1, 1) + timedelta(days=doy - 1, hours=hh, minutes=mm, seconds=ss)

def cloudsat_granule_to_xarray(fn):
    profile_time = read_vdata_by_ref(fn, 7, "Profile_time")

    # ✅ Correct time: t0 parsed from filename + Profile_time (seconds)
    t0 = np.datetime64(t0_from_filename(fn))
    time = t0 + np.round(profile_time * 1000).astype("timedelta64[ms]")

    lat = read_vdata_by_ref(fn, 10, "Latitude")
    lon = ((read_vdata_by_ref(fn, 11, "Longitude") + 180) % 360) - 180

    hdf = SD(fn, SDC.READ)
    cf, hgt, ucf = (hdf.select(k)[:] for k in ["CloudFraction", "Height", "UncertaintyCF"])
    lbase, ltop  = (hdf.select(k)[:] for k in ["LayerBase", "LayerTop"])
    dq, ds       = read_vdata_by_ref(fn, 18, "Data_quality"), read_vdata_by_ref(fn, 19, "Data_status")

    return xr.Dataset(
        data_vars=dict(
            CloudFraction=(("time","height_bin"), cf),
            Height=(("time","height_bin"), hgt),
            UncertaintyCF=(("time","height_bin"), ucf),
            LayerBase=(("time","cloud_layer"), lbase),
            LayerTop=(("time","cloud_layer"), ltop),
            Data_quality=("time", dq),
            Data_status=("time", ds),
            lat=("time", lat),
            lon=("time", lon),
        ),
        coords=dict(
            time=("time", time),
            height_bin=("height_bin", np.arange(cf.shape[1])),
            cloud_layer=("cloud_layer", np.arange(lbase.shape[1])),
        ),
        attrs=dict(
            source="CloudSat 2B-GEOPROF-LIDAR P2_R05",
            note="lat/lon stored as data variables; DimVal variables excluded",
            time_reference="t0 parsed from filename YYYYDOYHHMMSS; time=t0+Profile_time(seconds)",
        )
    )

def bbox_mask(lat, lon, bbox):
    latmin, latmax, lonmin, lonmax = bbox
    return (lat >= latmin) & (lat <= latmax) & (lon >= lonmin) & (lon <= lonmax)

def process_one_file(fn):
    # quick screening
    lat = read_vdata_by_ref(fn, 10, "Latitude")
    lon = ((read_vdata_by_ref(fn, 11, "Longitude") + 180) % 360) - 180
    m_ena, m_sgp = bbox_mask(lat, lon, ENA), bbox_mask(lat, lon, SGP)
    if not (m_ena.any() or m_sgp.any()):
        return 0, 0, False

    ds = cloudsat_granule_to_xarray(fn)
    base = os.path.basename(fn).replace(".hdf", ".nc")

    nena = nsgp = 0
    if m_ena.any():
        ds_ena = ds.isel(time=np.where(m_ena)[0])
        ds_ena.to_netcdf(os.path.join(OUT_ENA, base))
        nena = ds_ena.sizes["time"]

    if m_sgp.any():
        ds_sgp = ds.isel(time=np.where(m_sgp)[0])
        ds_sgp.to_netcdf(os.path.join(OUT_SGP, base))
        nsgp = ds_sgp.sizes["time"]

    return nena, nsgp, True

def main():
    files = sorted(glob.glob(os.path.join(SRC_ROOT, "*", "*", "*.hdf")))
    print("Total files:", len(files))

    ok = skipped = 0
    total_ena = total_sgp = 0

    for i, fn in enumerate(files, 1):
        try:
            nena, nsgp, did = process_one_file(fn)
            ok += 1
            total_ena += nena
            total_sgp += nsgp
            if did and (nena or nsgp):
                print(f"[{i}/{len(files)}] hit ENA={nena} SGP={nsgp} :: {fn}")
        except Exception as e:
            skipped += 1
            print(f"[{i}/{len(files)}] BAD FILE (skipped): {fn}")
            print("   ERROR:", repr(e))

    print("DONE")
    print("  processed_ok:", ok)
    print("  skipped_bad :", skipped)
    print("  rays_selected: ENA", total_ena, "SGP", total_sgp)

if __name__ == "__main__":
    main()


Total files: 51490
[19/51490] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-GEOPROF-LIDAR.P2_R05/2006/165/2006165023524_00681_CS_2B-GEOPROF-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[26/51490] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-GEOPROF-LIDAR.P2_R05/2006/165/2006165140736_00688_CS_2B-GEOPROF-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[65/51490] hit ENA=411 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-GEOPROF-LIDAR.P2_R05/2006/172/2006172024140_00783_CS_2B-GEOPROF-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[72/51490] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-GEOPROF-LIDAR.P2_R05/2006/172/2006172141352_00790_CS_2B-GEOPROF-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[94/51490] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-GEOPROF-LIDAR.P2_R05/2006/174/2006174022920_00812_CS_2B-GEOPROF-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[101/51490] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-GEOPROF-LIDAR.P2_R05/2006/174/200

In [9]:
import os, glob
import numpy as np
import xarray as xr
from pyhdf.SD import SD, SDC
from pyhdf.HDF import HDF
from datetime import datetime, timedelta

SRC_ROOT = "/data/shared_data/satellite_data/CloudSat/2B-CLDCLASS-LIDAR.P1_R05"
OUT_ENA  = "/data/ggong/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/ENA_5x5"
OUT_SGP  = "/data/ggong/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/SGP_5x5"

ENA = (36.5916, 41.5916, -30.5257, -25.5257)
SGP = (34.107322, 39.107322, -99.987643, -94.987643)

os.makedirs(OUT_ENA, exist_ok=True)
os.makedirs(OUT_SGP, exist_ok=True)

def read_vdata_by_ref(fn, ref, field):
    h = HDF(fn); vs = h.vstart()
    v = vs.attach(ref)
    nrecs, _, _, _, _ = v.inquire()
    v.setfields(field); raw = v.read(nrecs)
    v.detach(); vs.end(); h.close()
    arr = np.array(raw)
    return arr[:, 0] if (arr.ndim == 2 and arr.shape[1] == 1) else arr

def t0_from_filename(fn):
    # filename begins with YYYYDOYHHMMSS, e.g. 2006163205526
    stem = os.path.basename(fn)[:13]
    y, doy = int(stem[0:4]), int(stem[4:7])
    hh, mm, ss = int(stem[7:9]), int(stem[9:11]), int(stem[11:13])
    return datetime(y, 1, 1) + timedelta(days=doy - 1, hours=hh, minutes=mm, seconds=ss)

def bbox_mask(lat, lon, bbox):
    latmin, latmax, lonmin, lonmax = bbox
    return (lat >= latmin) & (lat <= latmax) & (lon >= lonmin) & (lon <= lonmax)

def cloudsat_granule_to_xarray(fn):
    # ✅ Correct time: t0 parsed from filename + Profile_time (seconds)
    profile_time = read_vdata_by_ref(fn, 7, "Profile_time")
    t0 = np.datetime64(t0_from_filename(fn))
    time = t0 + np.round(profile_time * 1000).astype("timedelta64[ms]")

    lat = read_vdata_by_ref(fn, 10, "Latitude")
    lon = ((read_vdata_by_ref(fn, 11, "Longitude") + 180) % 360) - 180

    hdf = SD(fn, SDC.READ)
    all_sds = list(hdf.datasets().keys())

    drop_sds = {"nray", "nbin", "ncloud", "nvfmavg"}
    keep = [name for name in all_sds if not any(k in name.lower() for k in drop_sds)]

    data_vars = {"lat": ("time", lat), "lon": ("time", lon)}

    for name in keep:
        a = hdf.select(name)[:]
        if a.ndim == 1:
            data_vars[name] = ("time", a)
        elif a.ndim == 2:
            data_vars[name] = (("time", f"{name}_dim1"), a)
        elif a.ndim == 3:
            data_vars[name] = (("time", f"{name}_dim1", f"{name}_dim2"), a)

    return xr.Dataset(
        data_vars=data_vars,
        coords=dict(time=("time", time)),
        attrs=dict(
            source="CloudSat 2B-CLDCLASS-LIDAR P1_R05",
            note="lat/lon stored as data variables; SDS auto-loaded; DimVal excluded",
            time_reference="t0 parsed from filename YYYYDOYHHMMSS; time=t0+Profile_time(seconds)",
        )
    )

def process_one_file(fn):
    lat = read_vdata_by_ref(fn, 10, "Latitude")
    lon = ((read_vdata_by_ref(fn, 11, "Longitude") + 180) % 360) - 180
    m_ena, m_sgp = bbox_mask(lat, lon, ENA), bbox_mask(lat, lon, SGP)
    if not (m_ena.any() or m_sgp.any()):
        return 0, 0, False

    ds = cloudsat_granule_to_xarray(fn)
    base = os.path.basename(fn).replace(".hdf", ".nc")

    nena = nsgp = 0
    if m_ena.any():
        ds_ena = ds.isel(time=np.where(m_ena)[0])
        ds_ena.to_netcdf(os.path.join(OUT_ENA, base))
        nena = ds_ena.sizes["time"]

    if m_sgp.any():
        ds_sgp = ds.isel(time=np.where(m_sgp)[0])
        ds_sgp.to_netcdf(os.path.join(OUT_SGP, base))
        nsgp = ds_sgp.sizes["time"]

    return nena, nsgp, True

def main():
    files = sorted(glob.glob(os.path.join(SRC_ROOT, "*", "*", "*.hdf")))
    print("Total files:", len(files))

    ok = skipped = 0
    total_ena = total_sgp = 0

    for i, fn in enumerate(files, 1):
        try:
            nena, nsgp, did = process_one_file(fn)
            ok += 1
            total_ena += nena
            total_sgp += nsgp
            if did and (nena or nsgp):
                print(f"[{i}/{len(files)}] hit ENA={nena} SGP={nsgp} :: {fn}")
        except Exception as e:
            skipped += 1
            print(f"[{i}/{len(files)}] BAD FILE (skipped): {fn}")
            print("   ERROR:", repr(e))

    print("DONE")
    print("  processed_ok:", ok)
    print("  skipped_bad :", skipped)
    print("  rays_selected: ENA", total_ena, "SGP", total_sgp)

if __name__ == "__main__":
    main()


Total files: 50671
[19/50671] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/2006/165/2006165023524_00681_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E00_F00.hdf
[26/50671] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/2006/165/2006165140736_00688_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E00_F00.hdf
[67/50671] hit ENA=411 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/2006/172/2006172024140_00783_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E00_F00.hdf
[74/50671] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/2006/172/2006172141352_00790_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E00_F00.hdf
[96/50671] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/2006/174/2006174022920_00812_CS_2B-CLDCLASS-LIDAR_GRANULE_P1_R05_E00_F00.hdf
[103/50671] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CLDCLASS-LIDAR.P1_R05/2

In [10]:
import os, glob
import numpy as np
import xarray as xr
from pyhdf.SD import SD, SDC
from pyhdf.HDF import HDF
from datetime import datetime, timedelta

SRC_ROOT = "/data/shared_data/satellite_data/CloudSat/2B-FLXHR-LIDAR.P2_R05"
OUT_ENA  = "/data/ggong/CloudSat/2B-FLXHR-LIDAR.P2_R05/ENA_5x5"
OUT_SGP  = "/data/ggong/CloudSat/2B-FLXHR-LIDAR.P2_R05/SGP_5x5"

ENA = (36.5916, 41.5916, -30.5257, -25.5257)
SGP = (34.107322, 39.107322, -99.987643, -94.987643)

os.makedirs(OUT_ENA, exist_ok=True)
os.makedirs(OUT_SGP, exist_ok=True)

def read_vdata_by_ref(fn, ref, field):
    h = HDF(fn); vs = h.vstart()
    v = vs.attach(ref)
    nrecs, _, _, _, _ = v.inquire()
    v.setfields(field); raw = v.read(nrecs)
    v.detach(); vs.end(); h.close()
    arr = np.array(raw)
    return arr[:, 0] if (arr.ndim == 2 and arr.shape[1] == 1) else arr

def t0_from_filename(fn):
    # filename begins with YYYYDOYHHMMSS, e.g. 2006165023524
    stem = os.path.basename(fn)[:13]
    y, doy = int(stem[0:4]), int(stem[4:7])
    hh, mm, ss = int(stem[7:9]), int(stem[9:11]), int(stem[11:13])
    return datetime(y, 1, 1) + timedelta(days=doy - 1, hours=hh, minutes=mm, seconds=ss)

def bbox_mask(lat, lon, bbox):
    latmin, latmax, lonmin, lonmax = bbox
    return (lat >= latmin) & (lat <= latmax) & (lon >= lonmin) & (lon <= lonmax)

def cloudsat_granule_to_xarray(fn):
    # ---- time ----
    profile_time = read_vdata_by_ref(fn, 7, "Profile_time")
    t0 = np.datetime64(t0_from_filename(fn))
    time = t0 + np.round(profile_time * 1000).astype("timedelta64[ms]")

    # ---- geolocation ----
    lat = read_vdata_by_ref(fn, 10, "Latitude")
    lon = ((read_vdata_by_ref(fn, 11, "Longitude") + 180) % 360) - 180

    # ---- Vdata you listed (exclude time/lat/lon + DimVal) ----
    # (ref_id, fieldname, out_varname)
    vlist = [
        (13, "Range_to_intercept", "Range_to_intercept"),
        (14, "DEM_elevation", "DEM_elevation"),
        (18, "SurfaceHeightBin", "SurfaceHeightBin"),
        (19, "Data_quality", "Data_quality"),
        (20, "Data_status", "Data_status"),
        (21, "Data_targetID", "Data_targetID"),
        (22, "RayStatus_validity", "RayStatus_validity"),
        (23, "Navigation_land_sea_flag", "Navigation_land_sea_flag"),
        (34, "FD_TOA_IncomingSolar", "FD_TOA_IncomingSolar"),
        (53, "Scene_status", "Scene_status"),
        (54, "Lidar_status", "Lidar_status"),
        (55, "Status", "Status"),
        (94, "Solar_zenith_angle", "Solar_zenith_angle"),
        (95, "Land_Char", "Land_Char"),
        (96, "Albedo", "Albedo"),
        (97, "FlagCounts", "FlagCounts"),
    ]

    data_vars = {
        "lat": ("time", lat),
        "lon": ("time", lon),
    }

    for ref, field, outname in vlist:
        arr = read_vdata_by_ref(fn, ref, field)
        if arr.ndim == 1 and arr.shape[0] == time.shape[0]:
            data_vars[outname] = ("time", arr)
        else:
            # e.g., FlagCounts is length 20, not time-length
            data_vars[outname] = (("dim_" + outname,), arr)

    # ---- SDS: auto-load all and drop DimVal dimension holders ----
    hdf = SD(fn, SDC.READ)
    all_sds = list(hdf.datasets().keys())
    drop_keys = ("nray", "nbin", "ncloud", "nvfmavg", "nbands", "nz_flux", "nz_hr", "nflags", "nlatbins")
    keep = [k for k in all_sds if not any(d in k.lower() for d in drop_keys)]

    for name in keep:
        a = hdf.select(name)[:]
        if a.ndim == 1 and a.shape[0] == time.shape[0]:
            data_vars[name] = ("time", a)
        elif a.ndim == 2 and a.shape[0] == time.shape[0]:
            data_vars[name] = (("time", f"{name}_dim1"), a)
        elif a.ndim == 3 and a.shape[0] == time.shape[0]:
            data_vars[name] = (("time", f"{name}_dim1", f"{name}_dim2"), a)

    return xr.Dataset(
        data_vars=data_vars,
        coords=dict(time=("time", time)),
        attrs=dict(
            source="CloudSat 2B-FLXHR-LIDAR P2_R05",
            note="lat/lon stored as data variables; Vdata+SDS loaded; DimVal variables excluded",
            time_reference="t0 parsed from filename YYYYDOYHHMMSS; time=t0+Profile_time(seconds)",
        )
    )

def process_one_file(fn):
    # quick bbox screening (read only lat/lon)
    lat = read_vdata_by_ref(fn, 10, "Latitude")
    lon = ((read_vdata_by_ref(fn, 11, "Longitude") + 180) % 360) - 180
    m_ena, m_sgp = bbox_mask(lat, lon, ENA), bbox_mask(lat, lon, SGP)
    if not (m_ena.any() or m_sgp.any()):
        return 0, 0, False

    ds = cloudsat_granule_to_xarray(fn)
    base = os.path.basename(fn).replace(".hdf", ".nc")

    nena = nsgp = 0
    if m_ena.any():
        ds_ena = ds.isel(time=np.where(m_ena)[0])
        ds_ena.to_netcdf(os.path.join(OUT_ENA, base))
        nena = ds_ena.sizes["time"]

    if m_sgp.any():
        ds_sgp = ds.isel(time=np.where(m_sgp)[0])
        ds_sgp.to_netcdf(os.path.join(OUT_SGP, base))
        nsgp = ds_sgp.sizes["time"]

    return nena, nsgp, True

def main():
    files = sorted(glob.glob(os.path.join(SRC_ROOT, "*", "*", "*.hdf")))
    print("Total files:", len(files))

    ok = skipped = 0
    total_ena = total_sgp = 0

    for i, fn in enumerate(files, 1):
        try:
            nena, nsgp, did = process_one_file(fn)
            ok += 1
            total_ena += nena
            total_sgp += nsgp
            if did and (nena or nsgp):
                print(f"[{i}/{len(files)}] hit ENA={nena} SGP={nsgp} :: {fn}")
        except Exception as e:
            skipped += 1
            print(f"[{i}/{len(files)}] BAD FILE (skipped): {fn}")
            print("   ERROR:", repr(e))

    print("DONE")
    print("  processed_ok:", ok)
    print("  skipped_bad :", skipped)
    print("  rays_selected: ENA", total_ena, "SGP", total_sgp)

if __name__ == "__main__":
    main()


Total files: 41836
[14/41836] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-FLXHR-LIDAR.P2_R05/2006/165/2006165023524_00681_CS_2B-FLXHR-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[19/41836] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-FLXHR-LIDAR.P2_R05/2006/165/2006165140736_00688_CS_2B-FLXHR-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[55/41836] hit ENA=411 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-FLXHR-LIDAR.P2_R05/2006/172/2006172024140_00783_CS_2B-FLXHR-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[62/41836] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-FLXHR-LIDAR.P2_R05/2006/172/2006172141352_00790_CS_2B-FLXHR-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[82/41836] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-FLXHR-LIDAR.P2_R05/2006/174/2006174022920_00812_CS_2B-FLXHR-LIDAR_GRANULE_P2_R05_E00_F00.hdf
[89/41836] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-FLXHR-LIDAR.P2_R05/2006/174/2006174140131_00819_CS_2B-

In [4]:
import os, glob
import numpy as np
import xarray as xr
from pyhdf.SD import SD, SDC
from pyhdf.HDF import HDF
from datetime import datetime, timedelta

SRC_ROOT = "/data/shared_data/satellite_data/CloudSat/ECMWF-AUX.P1_R05"
OUT_ENA  = "/data/ggong/CloudSat/ECMWF-AUX.P1_R05/ENA_5x5"
OUT_SGP  = "/data/ggong/CloudSat/ECMWF-AUX.P1_R05/SGP_5x5"

ENA = (36.5916, 41.5916, -30.5257, -25.5257)
SGP = (34.107322, 39.107322, -99.987643, -94.987643)

os.makedirs(OUT_ENA, exist_ok=True)
os.makedirs(OUT_SGP, exist_ok=True)

def read_vdata_by_ref(fn, ref, field):
    h = HDF(fn); vs = h.vstart()
    v = vs.attach(ref)
    nrecs, _, _, _, _ = v.inquire()
    v.setfields(field); raw = v.read(nrecs)
    v.detach(); vs.end(); h.close()
    arr = np.array(raw)
    return arr[:, 0] if (arr.ndim == 2 and arr.shape[1] == 1) else arr

def t0_from_filename(fn):
    # filename begins with YYYYDOYHHMMSS, e.g. 2006163205526
    stem = os.path.basename(fn)[:13]
    y, doy = int(stem[0:4]), int(stem[4:7])
    hh, mm, ss = int(stem[7:9]), int(stem[9:11]), int(stem[11:13])
    return datetime(y, 1, 1) + timedelta(days=doy - 1, hours=hh, minutes=mm, seconds=ss)

def bbox_mask(lat, lon, bbox):
    latmin, latmax, lonmin, lonmax = bbox
    return (lat >= latmin) & (lat <= latmax) & (lon >= lonmin) & (lon <= lonmax)

def cloudsat_granule_to_xarray(fn):
    # ---- time ----
    pt = read_vdata_by_ref(fn, 7, "Profile_time")
    t0 = np.datetime64(t0_from_filename(fn))
    time = t0 + np.round(pt * 1000).astype("timedelta64[ms]")

    # ---- geolocation ----
    lat = read_vdata_by_ref(fn, 10, "Latitude")
    lon = ((read_vdata_by_ref(fn, 11, "Longitude") + 180) % 360) - 180

    # ---- ECM vertical grid (125) ----
    ec_height = read_vdata_by_ref(fn, 6, "EC_height")

    # ---- 1D Vdata (time) ----
    dem  = read_vdata_by_ref(fn, 12, "DEM_elevation")
    skt  = read_vdata_by_ref(fn, 27, "Skin_temperature")
    sp   = read_vdata_by_ref(fn, 28, "Surface_pressure")
    t2m  = read_vdata_by_ref(fn, 29, "Temperature_2m")
    sst  = read_vdata_by_ref(fn, 30, "Sea_surface_temperature")
    u10  = read_vdata_by_ref(fn, 31, "U10_velocity")
    v10  = read_vdata_by_ref(fn, 32, "V10_velocity")

    # ---- SDS (profiles etc.) ----
    hdf = SD(fn, SDC.READ)
    sds_names = list(hdf.datasets().keys())

    # drop DimVal-like holders
    drop_keys = ("nray", "nbin")
    sds_names = [n for n in sds_names if not any(k in n.lower() for k in drop_keys)]

    data_vars = dict(
        lat=("time", lat),
        lon=("time", lon),
        EC_height=("ec_height", ec_height),
        DEM_elevation=("time", dem),
        Skin_temperature=("time", skt),
        Surface_pressure=("time", sp),
        Temperature_2m=("time", t2m),
        Sea_surface_temperature=("time", sst),
        U10_velocity=("time", u10),
        V10_velocity=("time", v10),
    )

    # SDS: only keep those whose first dim matches time
    for name in sds_names:
        a = hdf.select(name)[:]
        if a.ndim == 1 and a.shape[0] == time.shape[0]:
            data_vars[name] = ("time", a)
        elif a.ndim == 2 and a.shape[0] == time.shape[0]:
            # common case: (time, 125) -> use ec_height
            dim1 = "ec_height" if a.shape[1] == len(ec_height) else f"{name}_dim1"
            data_vars[name] = (("time", dim1), a)
        elif a.ndim == 3 and a.shape[0] == time.shape[0]:
            data_vars[name] = (("time", f"{name}_dim1", f"{name}_dim2"), a)

    return xr.Dataset(
        data_vars=data_vars,
        coords=dict(
            time=("time", time),
            ec_height=("ec_height", np.arange(len(ec_height))),
        ),
        attrs=dict(
            source="CloudSat ECMWF-AUX P1_R05",
            note="lat/lon in data_vars; Vdata listed loaded; SDS auto-loaded when first dim==time; DimVal excluded",
            time_reference="t0 parsed from filename YYYYDOYHHMMSS; time=t0+Profile_time(seconds)",
        )
    )

def process_one_file(fn):
    lat = read_vdata_by_ref(fn, 10, "Latitude")
    lon = ((read_vdata_by_ref(fn, 11, "Longitude") + 180) % 360) - 180
    m_ena, m_sgp = bbox_mask(lat, lon, ENA), bbox_mask(lat, lon, SGP)
    if not (m_ena.any() or m_sgp.any()):
        return 0, 0, False

    ds = cloudsat_granule_to_xarray(fn)
    base = os.path.basename(fn).replace(".hdf", ".nc")

    nena = nsgp = 0
    if m_ena.any():
        ds_ena = ds.isel(time=np.where(m_ena)[0])
        ds_ena.to_netcdf(os.path.join(OUT_ENA, base))
        nena = ds_ena.sizes["time"]

    if m_sgp.any():
        ds_sgp = ds.isel(time=np.where(m_sgp)[0])
        ds_sgp.to_netcdf(os.path.join(OUT_SGP, base))
        nsgp = ds_sgp.sizes["time"]

    return nena, nsgp, True

def main():
    files = sorted(glob.glob(os.path.join(SRC_ROOT, "*", "*", "*.hdf")))
    print("Total files:", len(files))

    ok = skipped = 0
    total_ena = total_sgp = 0

    for i, fn in enumerate(files, 1):
        try:
            nena, nsgp, did = process_one_file(fn)
            ok += 1
            total_ena += nena
            total_sgp += nsgp
            if did and (nena or nsgp):
                print(f"[{i}/{len(files)}] hit ENA={nena} SGP={nsgp} :: {fn}")
        except Exception as e:
            skipped += 1
            print(f"[{i}/{len(files)}] BAD FILE (skipped): {fn}")
            print("   ERROR:", repr(e))

    print("DONE")
    print("  processed_ok:", ok)
    print("  skipped_bad :", skipped)
    print("  rays_selected: ENA", total_ena, "SGP", total_sgp)

if __name__ == "__main__":
    main()


Total files: 63167
[4/63167] hit ENA=0 SGP=340 :: /data/shared_data/satellite_data/CloudSat/ECMWF-AUX.P1_R05/2006/153/2006153183902_00516_CS_ECMWF-AUX_GRANULE_P1_R05_E00_F00.hdf
[38/63167] hit ENA=432 SGP=0 :: /data/shared_data/satellite_data/CloudSat/ECMWF-AUX.P1_R05/2006/156/2006156024112_00550_CS_ECMWF-AUX_GRANULE_P1_R05_E00_F00.hdf
[45/63167] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/ECMWF-AUX.P1_R05/2006/156/2006156141325_00557_CS_ECMWF-AUX_GRANULE_P1_R05_E00_F00.hdf
[67/63167] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/ECMWF-AUX.P1_R05/2006/158/2006158022856_00579_CS_ECMWF-AUX_GRANULE_P1_R05_E00_F00.hdf
[74/63167] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/ECMWF-AUX.P1_R05/2006/158/2006158140108_00586_CS_ECMWF-AUX_GRANULE_P1_R05_E00_F00.hdf
[77/63167] hit ENA=0 SGP=299 :: /data/shared_data/satellite_data/CloudSat/ECMWF-AUX.P1_R05/2006/158/2006158185748_00589_CS_ECMWF-AUX_GRANULE_P1_R05_E00_F00.hdf
[99/63167] hit ENA=0 S

In [5]:
import os, glob
import numpy as np
import xarray as xr
from pyhdf.SD import SD, SDC
from pyhdf.HDF import HDF
from datetime import datetime, timedelta

SRC_ROOT = "/data/shared_data/satellite_data/CloudSat/2B-CWC-RVOD.P1_R05"
OUT_ENA  = "/data/ggong/CloudSat/2B-CWC-RVOD.P1_R05/ENA_5x5"
OUT_SGP  = "/data/ggong/CloudSat/2B-CWC-RVOD.P1_R05/SGP_5x5"

ENA = (36.5916, 41.5916, -30.5257, -25.5257)
SGP = (34.107322, 39.107322, -99.987643, -94.987643)

os.makedirs(OUT_ENA, exist_ok=True)
os.makedirs(OUT_SGP, exist_ok=True)

def read_vdata_by_ref(fn, ref, field):
    h = HDF(fn); vs = h.vstart()
    v = vs.attach(ref)
    nrecs, _, _, _, _ = v.inquire()
    v.setfields(field); raw = v.read(nrecs)
    v.detach(); vs.end(); h.close()
    arr = np.array(raw)
    return arr[:, 0] if (arr.ndim == 2 and arr.shape[1] == 1) else arr

def t0_from_filename(fn):
    s = os.path.basename(fn)[:13]  # YYYYDOYHHMMSS
    y, doy = int(s[:4]), int(s[4:7])
    hh, mm, ss = int(s[7:9]), int(s[9:11]), int(s[11:13])
    return datetime(y, 1, 1) + timedelta(days=doy - 1, hours=hh, minutes=mm, seconds=ss)

def bbox_mask(lat, lon, bbox):
    latmin, latmax, lonmin, lonmax = bbox
    return (lat >= latmin) & (lat <= latmax) & (lon >= lonmin) & (lon <= lonmax)

def cloudsat_granule_to_xarray(fn):
    # ---- time (filename t0 + Profile_time) ----
    pt = read_vdata_by_ref(fn, 85, "Profile_time")
    t0 = np.datetime64(t0_from_filename(fn))
    time = t0 + np.round(pt * 1000).astype("timedelta64[ms]")

    # ---- lat/lon ----
    lat = read_vdata_by_ref(fn, 67, "Latitude")
    lon = ((read_vdata_by_ref(fn, 73, "Longitude") + 180) % 360) - 180

    # ---- 1D Vdata you listed ----
    vlist = [
        (19,  "Data_quality", "Data_quality"),
        (25,  "Data_status", "Data_status"),
        (31,  "RayStatus_validity", "RayStatus_validity"),
        (37,  "Data_targetID", "Data_targetID"),
        (43,  "Navigation_land_sea_flag", "Navigation_land_sea_flag"),
        (49,  "DEM_elevation", "DEM_elevation"),
        (91,  "Range_to_intercept", "Range_to_intercept"),
        (200, "Liq_Water_Path", "Liq_Water_Path"),
        (209, "Liq_Water_Path_Uncert", "Liq_Water_Path_Uncert"),
        (218, "Cloud_Liq_Water_Path", "Cloud_Liq_Water_Path"),
        (227, "Precip_Liq_Water_Path", "Precip_Liq_Water_Path"),
        (256, "Ice_Water_Path", "Ice_Water_Path"),
        (265, "Ice_Water_Path_Uncert", "Ice_Water_Path_Uncert"),
        (294, "PIA_Fwd", "PIA_Fwd"),
        (303, "Error_Flag", "Error_Flag"),
        (309, "Warning_Flag", "Warning_Flag"),
    ]

    data_vars = {"lat": ("time", lat), "lon": ("time", lon)}

    for ref, field, outname in vlist:
        arr = read_vdata_by_ref(fn, ref, field)
        if arr.ndim == 1 and arr.shape[0] == time.shape[0]:
            data_vars[outname] = ("time", arr)
        else:
            data_vars[outname] = ((f"dim_{outname}",), arr)

    # ---- SDS auto-load (skip DimVal like Nray/Nbin) ----
    hdf = SD(fn, SDC.READ)
    sds_names = list(hdf.datasets().keys())
    drop_keys = ("nray", "nbin")  # covers Nray/Nbin too
    sds_names = [n for n in sds_names if not any(k in n.lower() for k in drop_keys)]

    for name in sds_names:
        a = hdf.select(name)[:]
        if a.ndim == 1 and a.shape[0] == time.shape[0]:
            data_vars[name] = ("time", a)
        elif a.ndim == 2 and a.shape[0] == time.shape[0]:
            data_vars[name] = (("time", f"{name}_dim1"), a)
        elif a.ndim == 3 and a.shape[0] == time.shape[0]:
            data_vars[name] = (("time", f"{name}_dim1", f"{name}_dim2"), a)

    return xr.Dataset(
        data_vars=data_vars,
        coords=dict(time=("time", time)),
        attrs=dict(
            source="CloudSat 2B-CWC-RVOD P1_R05",
            note="lat/lon in data_vars; listed Vdata loaded; SDS auto-loaded when first dim==time; DimVal excluded",
            time_reference="t0 parsed from filename YYYYDOYHHMMSS; time=t0+Profile_time(seconds)",
        )
    )

def process_one_file(fn):
    # quick bbox screening
    lat = read_vdata_by_ref(fn, 67, "Latitude")
    lon = ((read_vdata_by_ref(fn, 73, "Longitude") + 180) % 360) - 180
    m_ena, m_sgp = bbox_mask(lat, lon, ENA), bbox_mask(lat, lon, SGP)
    if not (m_ena.any() or m_sgp.any()):
        return 0, 0, False

    ds = cloudsat_granule_to_xarray(fn)
    base = os.path.basename(fn).replace(".hdf", ".nc")

    nena = nsgp = 0
    if m_ena.any():
        ds_ena = ds.isel(time=np.where(m_ena)[0])
        ds_ena.to_netcdf(os.path.join(OUT_ENA, base))
        nena = ds_ena.sizes["time"]

    if m_sgp.any():
        ds_sgp = ds.isel(time=np.where(m_sgp)[0])
        ds_sgp.to_netcdf(os.path.join(OUT_SGP, base))
        nsgp = ds_sgp.sizes["time"]

    return nena, nsgp, True

def main():
    files = sorted(glob.glob(os.path.join(SRC_ROOT, "*", "*", "*.hdf")))
    print("Total files:", len(files))

    ok = skipped = 0
    total_ena = total_sgp = 0

    for i, fn in enumerate(files, 1):
        try:
            nena, nsgp, did = process_one_file(fn)
            ok += 1
            total_ena += nena
            total_sgp += nsgp
            if did and (nena or nsgp):
                print(f"[{i}/{len(files)}] hit ENA={nena} SGP={nsgp} :: {fn}")
        except Exception as e:
            skipped += 1
            print(f"[{i}/{len(files)}] BAD FILE (skipped): {fn}")
            print("   ERROR:", repr(e))

    print("DONE")
    print("  processed_ok:", ok)
    print("  skipped_bad :", skipped)
    print("  rays_selected: ENA", total_ena, "SGP", total_sgp)

if __name__ == "__main__":
    main()


Total files: 40491
[19/40491] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CWC-RVOD.P1_R05/2006/165/2006165023524_00681_CS_2B-CWC-RVOD_GRANULE_P1_R05_E00_F00.hdf
[26/40491] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CWC-RVOD.P1_R05/2006/165/2006165140736_00688_CS_2B-CWC-RVOD_GRANULE_P1_R05_E00_F00.hdf
[66/40491] hit ENA=411 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CWC-RVOD.P1_R05/2006/172/2006172024140_00783_CS_2B-CWC-RVOD_GRANULE_P1_R05_E00_F00.hdf
[73/40491] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CWC-RVOD.P1_R05/2006/172/2006172141352_00790_CS_2B-CWC-RVOD_GRANULE_P1_R05_E00_F00.hdf
[93/40491] hit ENA=523 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CWC-RVOD.P1_R05/2006/174/2006174022920_00812_CS_2B-CWC-RVOD_GRANULE_P1_R05_E00_F00.hdf
[100/40491] hit ENA=522 SGP=0 :: /data/shared_data/satellite_data/CloudSat/2B-CWC-RVOD.P1_R05/2006/174/2006174140131_00819_CS_2B-CWC-RVOD_GRANULE_P1_R05_E00_F00.